# 🧠 Aula 05 — Protocolos de Redes e Interação com GPUs

**Objetivo:** aplicar conceitos de redes (IPv4/IPv6, TCP/UDP) para configurar e operar
ambientes distribuídos com GPUs, garantindo comunicação eficiente em projetos de IA.

**Roteiro deste notebook:**
1. Verificação do ambiente (interfaces e sockets).
2. Teoria: IPv4, IPv6 e o modelo OSI.
3. Demo: TCP vs. UDP no loopback.
4. Atividade: servidor de telemetria de GPU via TCP.
5. Bônus: IPv4 vs. IPv6 na prática (sockets e resolução de nomes).
6. Discussão e síntese.

> 💡 **Tudo roda offline** no loopback (127.0.0.1): não é preciso internet nem duas
> máquinas. Em máquinas diferentes, basta trocar o IP.

## 1. Verificação do Ambiente

Vamos identificar a máquina e conferir se os módulos de rede do Python estão disponíveis.
No Colab/Linux, também vemos os comandos de terminal equivalentes.

In [ ]:
# @title 🔍 Ambiente de rede
# ============================================================================
# OBJETIVO: saber em que máquina estamos e qual o nome do host (hostname),
# que é o endereço pelo qual outros nós nos encontram na rede.
# ============================================================================
import socket

print(f"Hostname : {socket.gethostname()}")
# gethostbyname('') resolve o nome local para o IP desta máquina.
print(f"IP local : {socket.gethostbyname(socket.gethostname())}")
print()
print("No Colab/Linux, os mesmos dados saem com:")
print("  !ip -4 addr show   # IPv4")
print("  !ip -6 addr show   # IPv6")

## 2. Teoria: IPv4, IPv6 e o modelo OSI

| Versão | Tamanho | Notação | Observação |
| :--- | :--- | :--- | :--- |
| **IPv4** | 32 bits | `192.168.1.100` | Esgotado; NAT compensa |
| **IPv6** | 128 bits | `2001:db8::1` | Autoconfiguração; crescendo em data centers |

Um dado desce a **pilha OSI** (simplificada) na saída e sobe no destino:

| Camada | Protocolos | Papel |
| :--- | :--- | :--- |
| **Aplicação (7)** | HTTP, SSH, gRPC, NCCL | O que a aplicação de IA usa |
| **Transporte (4)** | TCP, UDP | Controle de entrega |
| **Rede (3)** | IPv4, IPv6, ICMP | Endereçamento e roteamento |
| **Enlace (2)** | Ethernet, InfiniBand | Comunicação física entre nós |

**TCP vs. UDP:** TCP garante entrega e ordem (SSH, datasets, modelos); UDP é rápido mas
sem garantia (telemetria, streaming).

## 3. Demo: TCP vs. UDP

Rodamos servidor e cliente de cada protocolo em threads separadas, no **loopback**.

- **TCP** é um fluxo de bytes: confirmamos que **todos os bytes** chegaram.
- **UDP** dispara datagramas sem confirmar; medimos quantos o servidor recebeu.

In [ ]:
# @title ⚡ TCP vs. UDP (confiabilidade x velocidade)
# ============================================================================
# OBJETIVO: comparar os dois protocolos de transporte na prática.
# TCP confirma a entrega; UDP só dispara (sem garantia).
# ============================================================================
import socket, threading, time

HOST = "127.0.0.1"        # loopback: a máquina conversa consigo mesma
PORTA_TCP, PORTA_UDP = 65432, 65433
MENSAGENS = 200

# ── TCP ─────────────────────────────────────────────────────────────────────
def servidor_tcp(resultado):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORTA_TCP)); s.listen()
        conn, _ = s.accept()
        with conn:
            total = 0
            while True:
                dados = conn.recv(1024)
                if not dados: break          # cliente fechou
                total += len(dados)
            conn.sendall(str(total).encode())  # confirma o total recebido
    resultado.append(total)

def cliente_tcp():
    time.sleep(0.2)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORTA_TCP))          # handshake de 3 vias
        enviados = 0
        for i in range(MENSAGENS):
            dado = f"msg-{i}".encode(); s.sendall(dado); enviados += len(dado)
        s.shutdown(socket.SHUT_WR)
        recebidos = int(s.recv(1024).decode())
    print(f"  TCP: {recebidos}/{enviados} bytes entregues (garantia de entrega)")

# ── UDP ─────────────────────────────────────────────────────────────────────
def servidor_udp(resultado):
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.bind((HOST, PORTA_UDP)); s.settimeout(1.0)
        recebidas = 0
        try:
            while True:
                s.recvfrom(1024); recebidas += 1   # 1 recvfrom = 1 datagrama
        except socket.timeout:
            pass
    resultado.append(recebidas)

def cliente_udp():
    time.sleep(0.2)
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        for i in range(MENSAGENS):
            s.sendto(f"msg-{i}".encode(), (HOST, PORTA_UDP))
    print(f"  UDP: {MENSAGENS} datagramas disparados (sem confirmar)")

# ── Execução ────────────────────────────────────────────────────────────────
print("[TCP]")
res = []
t1 = threading.Thread(target=servidor_tcp, args=(res,))
t2 = threading.Thread(target=cliente_tcp)
t1.start(); t2.start(); t1.join(); t2.join()

print("[UDP]")
res_u = []
t3 = threading.Thread(target=servidor_udp, args=(res_u,))
t4 = threading.Thread(target=cliente_udp)
t3.start(); t4.start(); t3.join(); t4.join()
print(f"  UDP: {res_u[0]}/{MENSAGENS} recebidas pelo servidor")

## 4. Atividade: servidor de telemetria de GPU (TCP)

Agora o cenário real: cada **nó de GPU** envia suas métricas (temperatura, VRAM,
utilização) em **JSON via TCP** para um servidor central. Como a ordem importa, TCP é a
escolha — é o mesmo padrão usado na sincronização de gradientes.

In [ ]:
# @title 📡 Servidor de telemetria + nó de GPU enviando métricas
# ============================================================================
# OBJETIVO: simular o envio de métricas de GPU para um servidor central via
# TCP. O servidor roda numa thread; o cliente envia o JSON e fecha.
# ============================================================================
import json, socket, threading, time

HOST, PORTA = "127.0.0.1", 9999

def ler_gpu():
    """Lê a GPU real (nvidia-smi); sem GPU, devolve um exemplo simulado."""
    try:
        import shutil, subprocess
        if shutil.which("nvidia-smi"):
            saida = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,temperature.gpu,memory.used,"
                 "memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
                capture_output=True, text=True).stdout.strip().splitlines()[0]
            nome, temp, usada, total, util = [p.strip() for p in saida.split(",")]
            return {"nome": nome, "temp_c": int(temp),
                    "vram_usada_gb": round(int(usada)/1024, 1),
                    "vram_total_gb": round(int(total)/1024, 1),
                    "utilizacao_pct": int(util)}
    except Exception:
        pass
    return {"nome": "NVIDIA Tesla T4 (simulado)", "temp_c": 72,
            "vram_usada_gb": 11.3, "vram_total_gb": 16.0, "utilizacao_pct": 87}

def servidor():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind((HOST, PORTA)); srv.listen(5)
        conn, addr = srv.accept()
        with conn:
            metricas = json.loads(conn.recv(4096).decode())
            print(f"Métricas recebidas de {addr[0]}:{addr[1]}")
            print(f"  GPU : {metricas['nome']}")
            print(f"  Temp: {metricas['temp_c']} C")
            print(f"  VRAM: {metricas['vram_usada_gb']}/{metricas['vram_total_gb']} GB")
            print(f"  Uso : {metricas['utilizacao_pct']}%")

def cliente_gpu():
    time.sleep(0.3)                          # aguarda o servidor subir
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as cli:
        cli.connect((HOST, PORTA))
        cli.send(json.dumps(ler_gpu()).encode())

t_srv = threading.Thread(target=servidor)
t_cli = threading.Thread(target=cliente_gpu)
t_srv.start(); t_cli.start(); t_srv.join(); t_cli.join()

## 5. Bônus: IPv4 vs. IPv6 na prática

Cada protocolo tem sua **família de socket** (`AF_INET` para IPv4, `AF_INET6` para IPv6).
Com **dual stack**, a máquina usa os dois ao mesmo tempo. A resolução de nomes
(`getaddrinfo`) revela os endereços de cada família.

In [ ]:
# @title 🌐 IPv4 vs. IPv6: famílias de socket e resolução
# ============================================================================
# OBJETIVO: ver as duas famílias de socket e resolver um nome para IPv4 e IPv6.
# Sem internet, a resolução apenas não encontra nada (não quebra).
# ============================================================================
import socket

s4 = socket.socket(socket.AF_INET,  socket.SOCK_STREAM)
s6 = socket.socket(socket.AF_INET6, socket.SOCK_STREAM)
print(f"Socket IPv4: {s4.family.name}")
print(f"Socket IPv6: {s6.family.name}")
s4.close(); s6.close()

print("\nResolução de www.google.com:")
try:
    for familia, _, _, _, endereco in socket.getaddrinfo("www.google.com", 80):
        print(f"  {familia.name:8s} -> {endereco[0]}")
except socket.gaierror as erro:
    print(f"  Sem resolução ({erro}) — provavelmente sem internet.")
    print("  Em data centers, os nós costumam ser dual stack (IPv4 + IPv6).")

## 6. Discussão em Grupo

Em grupos de 3–4, analisem o cenário do cluster de GPUs:

1. 4 nós com GPUs em Ethernet 1GbE; o treino está lento. É a rede? Qual *upgrade*?
2. Por que o PyTorch DDP usa TCP e não UDP para sincronizar gradientes?
3. Dataset de 500 GB: scp, rsync comprimido ou tar primeiro? Qual é mais rápido?
4. Na nuvem, IPv4 privado (10.x.x.x) vs. IPv6 para a comunicação entre nós?

> Atividade de pesquisa completa em `aulas/aula05/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) TCP ou UDP?** Escolha e justifique em uma frase: *(a)* transferir 200 GB de dataset;
*(b)* enviar telemetria de temperatura da GPU a cada segundo; *(c)* exibir um vídeo ao vivo.

**2) O handshake.** Descreva as três etapas do handshake TCP (SYN / SYN-ACK / ACK). Por que o
UDP **não** faz isso?

**3) Medindo a diferença.** Na célula-esqueleto, rode o envio de N mensagens por UDP e conte
quantas o servidor recebeu. Mude N e veja se há perdas no loopback. Explique o resultado.

**4) IPv4 × IPv6.** Qual a diferença de endereçamento e por que os data centers de IA estão
migrando (ou usando os dois ao mesmo tempo)?

**5) Operar o servidor.** Você precisa atualizar um dataset no servidor de GPUs sem
recomeçar do zero se a conexão cair. Qual ferramenta usar (`scp` ou `rsync`) e com quais
opções? Justifique.


In [ ]:
# @title Exercício 3 — quantas mensagens UDP chegam?
# ============================================================================
# OBJETIVO: disparar N datagramas UDP e contar quantos o servidor recebe.
# ============================================================================
import socket, threading, time

HOST, PORTA = "127.0.0.1", 65444
N = 500

recebidas = []

def servidor():
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.bind((HOST, PORTA)); s.settimeout(1.0)
        contador = 0
        try:
            while True:
                s.recvfrom(1024); contador += 1
        except socket.timeout:
            pass
        recebidas.append(contador)

def cliente():
    time.sleep(0.2)
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        for i in range(N):
            s.sendto(f"msg-{i}".encode(), (HOST, PORTA))

t1 = threading.Thread(target=servidor)
t2 = threading.Thread(target=cliente)
t1.start(); t2.start(); t1.join(); t2.join()
print(f"Enviadas: {N} | Recebidas: {recebidas[0]}")
print("No loopback quase tudo chega; numa rede real, o UDP pode perder pacotes.")

## 8. Síntese e Tarefa de Casa

**O que levar:**
- **IPv4:** 32 bits, dominante em LANs; **IPv6:** 128 bits, autoconfiguração.
- **TCP:** confiável, handshake 3 vias → SSH, transfers, APIs.
- **UDP:** rápido, sem garantia → telemetria, streaming.
- **SSH/scp/rsync:** operar GPUs remotas e transferir datasets (rsync é retomável).
- **Wireshark/netcat:** diagnosticar e simular tráfego.

**Tarefa (opcional):** configure uma mini rede local virtual (2 VMs ou 2 instâncias na
nuvem) e pratique:
- IPs estáticos em ambas;
- `ping` e `traceroute`;
- transferir um arquivo `.pt` via `scp` e `rsync`;
- um servidor TCP em Python recebendo métricas de GPU.

> 🔗 **Próxima aula:** *Sistemas Operacionais Linux e GPU* — o nó remoto precisa
> sustentar o hardware por horas: `/proc`, `/sys`, `tmux` e agendamento com `cron`.